# Step 3 — Feature Engineering
MIG Cement Demand Forecasting

Per the project spec: *"Create lag features, rolling aggregates, and
interaction variables. Engineer weather-adjusted pour indicators and
calculate inventory turnover metrics to capture site-specific operational
characteristics for modeling."*

Every feature here is justified by a specific EDA finding (`02_eda.ipynb`),
not added generically:

| EDA finding | Feature response |
|---|---|
| `behavior` is the dominant signal (65% vs 14% stockout-risk) | Keep as categorical; add interaction terms |
| Heavy rain (15mm+) crashes consumption non-linearly | Threshold flag, not raw `rain_mm` |
| `avg_temp_c` has ~0 correlation | Dropped — no signal |
| Planned-vs-actual adherence varies sharply by behavior (0.485–0.983) | Lag/smooth planned pour instead of using it raw |
| No day-of-week effect | No day-of-week feature |
| Series is stationary (ADF p≈0) | No differencing feature needed for tree models |
| Region is a weak, small-sample signal | Keep as filter, not a primary feature |

Reads `data/processed/operations_clean.parquet` (Step 1 output), writes
`data/processed/model_features.parquet` for Step 4.

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.width", 120)

IN_PATH = Path("../data/processed/operations_clean.parquet")
df = pd.read_parquet(IN_PATH)
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(["site_id", "date"]).reset_index(drop=True)  # NOT cement_type — see grain note in Section 1
print(df.shape)
df.head()

(32880, 17)


,date,site_id,cement_type,planned_pour_tonnes,consumed_tonnes,opening_inventory_tonnes,deliveries_tonnes,closing_inventory_tonnes,rain_mm,avg_temp_c,silo_capacity,region,behavior,overflow_tonnes,closing_inventory_raw_tonnes,stockout_risk,pour_shortfall_tonnes
0,2022-01-01,SITE_001,CEM_II,43.18,34.54,52.56,45.83,63.85,3.40,-3.10,448,North,aggressive,0.0,63.85,True,8.64
1,2022-01-02,SITE_001,CEM_I,45.26,45.26,63.85,19.97,38.56,3.23,14.28,448,North,aggressive,0.0,38.56,False,0.00
2,2022-01-03,SITE_001,CEM_III,38.69,38.69,38.56,47.19,47.06,2.64,6.40,448,North,aggressive,0.0,47.06,False,0.00
3,2022-01-04,SITE_001,CEM_I,33.16,33.16,47.06,18.74,32.64,8.25,14.23,448,North,aggressive,0.0,32.64,False,0.00
4,2022-01-05,SITE_001,CEM_III,56.88,47.04,32.64,14.40,0.00,2.69,8.97,448,North,aggressive,0.0,0.00,True,9.84


## 1. Lag features

Lags of the target (`consumed_tonnes`) let tree-based models (Step 4:
Random Forest / LightGBM) see recent history without needing explicit
AR terms the way SARIMAX does natively.

**Grouping key — important**: each site logs exactly **one row per
calendar day** (confirmed: every site has exactly 1,096 rows = the full
date range). `cement_type` is a same-day categorical attribute that
rotates day to day — it is **not** a parallel concurrent series. Grouping
by `(site_id, cement_type)` would fragment each site's continuous daily
series into three irregularly-spaced sub-series (a `consumed_lag_1d`
computed that way means "the last day this site happened to use this
cement type," which could be several calendar days earlier, not
yesterday). Lags/rolling features below are grouped by `site_id` alone,
sorted by date — a true continuous daily series per site.

Horizons chosen to match the 8-week forecast target: short lags (1, 7
days) for immediate momentum, plus 14/28-day lags for medium-term
pattern, since the model needs to extrapolate up to 56 days ahead.

**Note on "site-specific operational characteristics" (doc requirement)**:
this is satisfied by grouping every lag/rolling feature below by
`site_id` — each site's features reflect its own history, not a group
average. A one-hot `site_id` column (30 dummies) was deliberately **not**
added: with only 3 years of daily data per site, 30 sparse indicator
columns risk overfitting far more than they help, versus the lag/rolling
features already carrying each site's individual pattern.

In [2]:
LAG_DAYS = [1, 7, 14, 28]

group = df.groupby("site_id")  # site_id ONLY — see note above on grain
for lag in LAG_DAYS:
    df[f"consumed_lag_{lag}d"] = group["consumed_tonnes"].shift(lag)

df[[f"consumed_lag_{l}d" for l in LAG_DAYS]].isnull().sum()

consumed_lag_1d      30
consumed_lag_7d     210
consumed_lag_14d    420
consumed_lag_28d    840
dtype: int64

Note: the first 28 rows of each site's series will have nulls in the
longest lag column — this is expected and correct (there's no history
before day 1). With grouping now correctly by `site_id` alone, this is
28 rows x 30 sites = max 840 rows, not the ~2,520 the old (site_id,
cement_type) grouping would have produced.

## 2. Rolling aggregates

Smoothed recent-history features — rolling mean and std over trailing
windows. `min_periods=1` so early rows still get a value (from whatever
history exists) rather than propagating nulls further than necessary.

In [3]:
ROLLING_WINDOWS = [7, 14, 28]

for window in ROLLING_WINDOWS:
    df[f"consumed_roll_mean_{window}d"] = (
        group["consumed_tonnes"]
        .transform(lambda s: s.shift(1).rolling(window, min_periods=1).mean())
    )
    df[f"consumed_roll_std_{window}d"] = (
        group["consumed_tonnes"]
        .transform(lambda s: s.shift(1).rolling(window, min_periods=1).std())
    )

# shift(1) before rolling is deliberate: without it, the rolling window
# for day N would include day N's own consumption — a direct target leak.
# This is the single most common bug in time-series feature engineering.
df[[c for c in df.columns if "roll" in c]].describe().T[["mean", "std", "min", "max"]]

,mean,std,min,max
consumed_roll_mean_7d,23.734566,9.692604,0.000000,64.290000
consumed_roll_std_7d,12.991671,7.299792,0.106066,41.959716
consumed_roll_mean_14d,23.745833,8.921803,0.000000,64.290000
consumed_roll_std_14d,13.236774,6.754076,0.106066,41.959716
consumed_roll_mean_28d,23.761757,8.543074,0.000000,64.290000
consumed_roll_std_28d,13.323660,6.507722,0.106066,41.959716


## 3. Weather-adjusted pour indicators

Directly responds to the EDA finding that raw `rain_mm` has almost no
*linear* correlation with consumption (-0.18) but hides a sharp threshold
effect: heavy rain (15mm+) crashes consumption to ~15% of normal while
planned pours stay unchanged. A linear rain term would miss this
entirely — the threshold flag captures it directly.

In [4]:
# Threshold bands, informed directly by the EDA rain-band analysis
df["heavy_rain_flag"] = (df["rain_mm"] >= 15).astype(int)
df["moderate_rain_flag"] = ((df["rain_mm"] >= 5) & (df["rain_mm"] < 15)).astype(int)

# Frost/freeze flag — separate mechanism from rain (affects concrete curing
# time, not pour logistics), even though avg_temp_c showed no *linear*
# correlation with consumption. Kept as a domain-driven feature regardless,
# since construction-industry knowledge (concrete cure time is temperature-
# sensitive below ~5°C) outweighs a null linear-correlation result here —
# EDA found no signal in this specific synthetic dataset, but the real-world
# mechanism is well established and worth keeping for when MIG's live data
# replaces this synthetic set.
df["frost_risk_flag"] = (df["avg_temp_c"] <= 5).astype(int)

# Weather-adjusted pour feasibility: planned pour scaled down under bad
# weather — a single interaction feature the model can use directly rather
# than learning the interaction from raw rain_mm x planned_pour_tonnes itself.
df["weather_adjusted_planned_pour"] = df["planned_pour_tonnes"] * np.where(
    df["heavy_rain_flag"] == 1, 0.3,
    np.where(df["moderate_rain_flag"] == 1, 0.85, 1.0)
)

df[["rain_mm", "heavy_rain_flag", "moderate_rain_flag", "frost_risk_flag",
    "planned_pour_tonnes", "weather_adjusted_planned_pour"]].sample(5, random_state=1)

,rain_mm,heavy_rain_flag,moderate_rain_flag,frost_risk_flag,planned_pour_tonnes,weather_adjusted_planned_pour
23370,4.55,0,0,1,35.86,35.86
23334,0.24,0,0,1,47.58,47.58
4077,0.67,0,0,0,9.64,9.64
31455,3.73,0,0,0,0.00,0.00
10976,0.68,0,0,0,36.11,36.11


The 0.3 / 0.85 scaling factors above are a **starting assumption**, not
derived from data — flag this explicitly in project documentation. Refine
them once real pour-cancellation data is available (e.g. if MIG logs
"pour postponed due to weather" as a reason code, fit these factors from
that directly rather than guessing).

## 4. Planned-pour reliability features

Responds to the EDA finding that planned-vs-actual correlation varies
sharply by behavior (aggressive 0.485 vs conservative 0.983) — raw
`planned_pour_tonnes` should not be trusted uniformly as a forecast input.
These features let the model learn how much to trust the plan, rather
than assuming it's always reliable.

In [5]:
# Rolling adherence ratio: how well has this site recently hit its plan?
df["planned_actual_ratio"] = (
    df["consumed_tonnes"] / df["planned_pour_tonnes"].replace(0, np.nan)
)
df["adherence_roll_mean_14d"] = (
    group["planned_actual_ratio"]
    .transform(lambda s: s.shift(1).rolling(14, min_periods=3).mean())
)

# Lagged planned pour — tomorrow's plan is known today in real operations
# (pour schedules are set in advance), so using it directly isn't leakage,
# but smoothing it against recent adherence gives the model a trust-adjusted
# version rather than the raw schedule number.
df["planned_pour_trust_adjusted"] = (
    df["planned_pour_tonnes"] * df["adherence_roll_mean_14d"].fillna(df["adherence_roll_mean_14d"].median())
)

df[["behavior", "planned_actual_ratio", "adherence_roll_mean_14d",
    "planned_pour_trust_adjusted"]].groupby(df["behavior"]).mean(numeric_only=True).round(3)

,planned_actual_ratio,adherence_roll_mean_14d,planned_pour_trust_adjusted
behavior,,,
aggressive,0.716,0.718,30.714
chaotic,0.886,0.888,27.352
conservative,0.970,0.970,11.518


## 5. Inventory turnover metrics

Feeds directly into Step 5 (inventory simulation / reorder points).
`days_of_stock` in particular is the basis for reorder-point logic —
how many days of buffer a site is currently carrying at its recent
consumption rate.

In [6]:
df["inventory_turnover_ratio"] = (
    df["consumed_tonnes"] / df["opening_inventory_tonnes"].replace(0, np.nan)
)

df["days_of_stock"] = (
    df["closing_inventory_tonnes"] /
    df["consumed_roll_mean_7d"].replace(0, np.nan)
)

df["silo_utilization_pct"] = df["closing_inventory_tonnes"] / df["silo_capacity"]

df[["days_of_stock", "silo_utilization_pct", "inventory_turnover_ratio"]].describe().T

,count,mean,std,min,25%,50%,75%,max
days_of_stock,32848.0,10.203382,14.304902,0.0,0.039879,1.854893,16.579405,160.478951
silo_utilization_pct,32880.0,0.430199,0.447591,0.0,0.004377,0.171065,1.000000,1.000000
inventory_turnover_ratio,24938.0,2.202201,27.801492,0.0,0.000889,0.034480,1.085960,2157.000000


## 6. Interaction variables

One-hot encode `behavior` and `cement_type` **first**, then build
interaction terms from the resulting numeric dummy columns. This matters
for two reasons:
1. SARIMAX (Step 4 baseline) needs numeric exogenous regressors — a raw
   string interaction column like `"aggressive_1"` isn't usable by it.
2. Genuine interactions must actually vary with the categorical level.
   An earlier draft of this notebook multiplied `planned_pour_tonnes` by
   the same constant (1.0) for every behavior — that's mathematically
   identical to `planned_pour_tonnes` itself, not an interaction. Caught
   and fixed here: the interactions below are computed as
   `continuous_feature x dummy_column`, so they're zero for rows outside
   that category and equal to the continuous value inside it — a
   standard, genuinely non-redundant way to let a linear model learn a
   different slope per category.

In [7]:
df = pd.get_dummies(df, columns=["behavior", "cement_type"], prefix=["behavior", "type"], drop_first=False)

behavior_dummy_cols = [c for c in df.columns if c.startswith("behavior_")]

# Real interactions: each is zero outside its category, non-zero inside it —
# lets SARIMAX (or any linear model) learn a per-behavior slope for the
# rain effect and the planned-pour effect, rather than one global slope.
for b_col in behavior_dummy_cols:
    behavior_name = b_col.replace("behavior_", "")
    df[f"heavy_rain_x_{behavior_name}"] = df["heavy_rain_flag"] * df[b_col]
    df[f"planned_pour_x_{behavior_name}"] = df["planned_pour_tonnes"] * df[b_col]

new_interaction_cols = [c for c in df.columns if c.startswith("heavy_rain_x_") or c.startswith("planned_pour_x_")]
print(df.shape)
print(new_interaction_cols)

# Sanity check the fix: unlike the old placeholder, these should NOT be
# identical to planned_pour_tonnes — confirm they differ (zero outside
# their category).
check = df[["behavior_aggressive", "planned_pour_tonnes", "planned_pour_x_aggressive"]].sample(8, random_state=2)
print()
print(check)

(32880, 47)
['heavy_rain_x_aggressive', 'planned_pour_x_aggressive', 'heavy_rain_x_chaotic', 'planned_pour_x_chaotic', 'heavy_rain_x_conservative', 'planned_pour_x_conservative']

       behavior_aggressive  planned_pour_tonnes  planned_pour_x_aggressive
25189                False                 5.22                       0.00
9491                 False                 0.00                       0.00
1259                 False                19.42                       0.00
5324                  True                38.94                      38.94
21193                 True                 0.00                       0.00
27743                False                48.15                       0.00
26192                False                49.69                       0.00
18631                 True                41.77                      41.77


## 7. Final feature set summary & save

In [8]:
feature_cols = [c for c in df.columns if c not in
                 ["date", "site_id", "region"]]  # keep identifiers/filters out of the numeric feature list

print(f"Total columns: {len(df.columns)}")
print(f"Rows: {len(df)}")

null_summary = df.isnull().sum()
null_summary = null_summary[null_summary > 0]

# Nulls have two genuinely different sources — don't lump them together:
lag_roll_cols = [c for c in null_summary.index if "lag" in c or "roll" in c]
ratio_cols = [c for c in null_summary.index if c not in lag_roll_cols]

print(f"\n(a) Lag/rolling history padding — expected, no history exists yet")
print(f"    (max 840 rows = 30 sites x 28-day longest lag, one continuous series per site):")
print(null_summary[lag_roll_cols])

print(f"\n(b) Zero-denominator days — NOT a bug, these are real 'no pour scheduled'")
print(f"    or 'zero opening stock' days where the ratio is structurally undefined:")
print(null_summary[ratio_cols])
print(f"    e.g. opening_inventory_tonnes==0 on {(df['opening_inventory_tonnes']==0).sum()} rows,")
print(f"    planned_pour_tonnes==0 on {(df['planned_pour_tonnes']==0).sum()} rows.")

OUT_PATH = Path("../data/processed/model_features.parquet")
df.to_parquet(OUT_PATH, index=False)
print(f"\nSaved -> {OUT_PATH} ({len(df)} rows, {len(df.columns)} columns)")

Total columns: 47
Rows: 32880

(a) Lag/rolling history padding — expected, no history exists yet
    (max 840 rows = 30 sites x 28-day longest lag, one continuous series per site):
consumed_lag_1d             30
consumed_lag_7d            210
consumed_lag_14d           420
consumed_lag_28d           840
consumed_roll_mean_7d       30
consumed_roll_std_7d        60
consumed_roll_mean_14d      30
consumed_roll_std_14d       60
consumed_roll_mean_28d      30
consumed_roll_std_28d       60
adherence_roll_mean_14d     98
dtype: int64

(b) Zero-denominator days — NOT a bug, these are real 'no pour scheduled'
    or 'zero opening stock' days where the ratio is structurally undefined:
planned_actual_ratio        2993
inventory_turnover_ratio    7942
days_of_stock                 32
dtype: int64
    e.g. opening_inventory_tonnes==0 on 7942 rows,
    planned_pour_tonnes==0 on 2993 rows.



Saved -> ../data/processed/model_features.parquet (32880 rows, 47 columns)


**Handoff note for Step 4 — two different null-handling decisions:**

1. **Lag/rolling padding nulls** (category a): drop for SARIMAX (can't
   handle NaNs), keep for LightGBM (handles natively, extra history is
   useful).
2. **Zero-denominator ratio nulls** (category b — `planned_actual_ratio`,
   `inventory_turnover_ratio`, `days_of_stock`): do **not** fill these with
   a mean/median — a null here means "no pour was scheduled" or "site had
   zero stock," which is real operational information, not missing data.
   Either leave as NaN for models that tolerate it, or engineer an
   explicit `had_zero_opening_stock` / `no_pour_scheduled` flag instead of
   imputing a fake ratio value.